# 5. Frozen text representations (GPU)

Every article must fit whole: token counts for every tokenizer, no article above the limit. Then the frozen vectors:
Qwen3-Embedding-8B and its small version Qwen3-Embedding-0.6B (same instruction + article, last token), Mistral-Small-24B
Base (4 bit, mean of the tokens, layers 27 and 40: step 7 chooses between them on the inner folds), and ModernBERT-base
used as it is (mean of its last layer), the frozen counterpart of the ModernBERT trained in step 7.

In [ ]:
QWEN = "Qwen/Qwen3-Embedding-8B"
QWEN_SMALL = "Qwen/Qwen3-Embedding-0.6B"
MISTRAL = "mistralai/Mistral-Small-24B-Base-2501"
BERT = "answerdotai/ModernBERT-base"
QWEN_BUDGET = 32768           # tokens per batch
MISTRAL_BUDGET = 8192         # smaller: all hidden states stay in memory
BERT_BUDGET = 32768
GPU_GB = 22

In [ ]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())

import models as MO

art, _ = MO.load()
texts = art.text.fillna("").tolist()
C.EMBEDDINGS.mkdir(parents=True, exist_ok=True)

In [ ]:
rows = []
for name, model, prefix in [("ModernBERT", BERT, ""), ("Qwen3-8B", QWEN, f"Instruct: {MO.QWEN_INSTRUCTION}\nQuery:"), ("Mistral-24B", MISTRAL, "")]:
    n = MO.count_tokens(texts, model, prefix)
    limit = MO.MAX_TOKENS + (64 if prefix else 0)
    rows.append({"model": name, "median": int(np.median(n)), "99%": int(np.percentile(n, 99)), "max": int(n.max()),
                 "limit": limit, "truncated": int((n > limit).sum())})
tokens = pd.DataFrame(rows).set_index("model"); tokens.to_csv(C.OUT / "token_lengths.csv")
assert tokens.truncated.sum() == 0
tokens

In [ ]:
for name, model in [("Qwen3-8B", QWEN), ("Qwen3-0.6B", QWEN_SMALL)]:
    f = C.EMBEDDINGS / MO.EMBEDDING_FILES[name]
    if not f.exists():
        MO.wait_for_gpu(GPU_GB, LOG)
        np.save(f, MO.qwen_embeddings(texts, model, QWEN_BUDGET, LOG))
files = MO.EMBEDDING_FILES["Mistral-24B"]
if not all((C.EMBEDDINGS / f).exists() for f in files.values()):
    MO.wait_for_gpu(GPU_GB, LOG)
    E = MO.mistral_embeddings(texts, MISTRAL, list(files), MISTRAL_BUDGET, LOG, C.EMBEDDINGS / "mistral_partial.npy")
    for layer, f in files.items():
        np.save(C.EMBEDDINGS / f, E[layer])
    (C.EMBEDDINGS / "mistral_partial.npy").unlink(missing_ok=True)
f = C.EMBEDDINGS / MO.EMBEDDING_FILES["ModernBERT-frozen"]
if not f.exists():
    MO.wait_for_gpu(GPU_GB, LOG)
    np.save(f, MO.encoder_embeddings(texts, BERT, BERT_BUDGET, LOG))
art[["article_id"]].to_csv(C.EMBEDDINGS / "article_order.csv", index=False)
for name, files in MO.EMBEDDING_FILES.items():
    for file in (files.values() if isinstance(files, dict) else [files]):
        E = np.load(C.EMBEDDINGS / file, mmap_mode="r"); print(name, file, E.shape, "finite:", np.isfinite(E).all())